In [74]:
import pandas as pd
import numpy as np

In [75]:
df = pd.read_csv("AmesHousing.csv")
df.sample(2)

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
2648,2649,902128020,20,RM,60.0,7200,Pave,Grvl,Reg,Lvl,...,0,NaN,NaN,NaN,0,8,2006,COD,Normal,105000
404,405,527451320,160,RM,21.0,1680,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,5,2009,COD,Abnorml,112000


In [76]:
df.shape

(2930, 82)

In [77]:
# Drop Order and PID
df = df.drop(columns=['Order', 'PID'])
df.sample(2)

,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
1905,60,RL,100.0,9500,Pave,NaN,Reg,Lvl,AllPub,Corner,...,0,NaN,GdPrv,NaN,0,6,2007,WD,Normal,165000
1101,60,RL,NaN,11000,Pave,NaN,Reg,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,6,2008,WD,Normal,291000


In [78]:
missing_values_columns = df.isna().sum()
print(missing_values_columns[missing_values_columns > 0])

Lot Frontage       490
Alley             2732
Mas Vnr Type      1775
Mas Vnr Area        23
Bsmt Qual           80
Bsmt Cond           80
Bsmt Exposure       83
BsmtFin Type 1      80
BsmtFin SF 1         1
BsmtFin Type 2      81
BsmtFin SF 2         1
Bsmt Unf SF          1
Total Bsmt SF        1
Electrical           1
Bsmt Full Bath       2
Bsmt Half Bath       2
Fireplace Qu      1422
Garage Type        157
Garage Yr Blt      159
Garage Finish      159
Garage Cars          1
Garage Area          1
Garage Qual        159
Garage Cond        159
Pool QC           2917
Fence             2358
Misc Feature      2824
dtype: int64


In [79]:
df['Alley'] = df['Alley'].fillna('None')
df['Alley'].sample(4)

df['Fireplace Qu'] = df['Fireplace Qu'].fillna("None")
df['Fireplace Qu'].sample(4)

df['Fence'] = df['Fence'].fillna('None')
df['Fence'].sample(4)

df['Pool QC'] = df['Pool QC'].fillna("None")
df['Pool QC'].sample(4)

df['Misc Feature'] = df['Misc Feature'].fillna("None")
df['Misc Feature'].sample(4)

df['Mas Vnr Type'] = df['Mas Vnr Type'].fillna("None")
df['Mas Vnr Type'].sample(4)

# BsmtFin Type 2, BsmtFin Type 1, Bsmt Exposure, Bsmt Cond, Bsmt Qual
df['BsmtFin Type 2'] = df['BsmtFin Type 2'].fillna("None")
df['BsmtFin Type 2'].sample(4)

df['BsmtFin Type 1'] = df['BsmtFin Type 1'].fillna("None")
df['BsmtFin Type 1'].sample(4)

df['Bsmt Exposure'] = df['Bsmt Exposure'].fillna("None")
df['Bsmt Exposure'].sample(4)

df['Bsmt Cond'] = df['Bsmt Cond'].fillna("None")
df['Bsmt Cond'].sample(4)

df['Bsmt Qual'] = df['Bsmt Qual'].fillna("None")
df['Bsmt Qual'].sample(4)

df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])

# Fill the missing value of Total Basement SF with 0 moreover, 
# if the total Basement square feet is 0 then the Basement other features since they are dependent will also be set to none.
df['Total Bsmt SF'] = df['Total Bsmt SF'].fillna(0)
df.loc[
    (df['Total Bsmt SF'] == 0),
    ['BsmtFin Type 2', 'BsmtFin Type 1', 'Bsmt Exposure', 'Bsmt Cond', 'Bsmt Qual']
] = "None"
# BsmtFin SF 1, BsmtFin SF 2, Bsmt Unf SF
df.loc[
    (df['Total Bsmt SF'] == 0),
    ['BsmtFin SF 1', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Bsmt Full Bath', 'Bsmt Half Bath']
] = 0
df['Bsmt Full Bath'] = df['Bsmt Full Bath'].fillna(0)
df['Bsmt Half Bath'] = df['Bsmt Half Bath'].fillna(0)

# If Mansory Venue type is none then the area is subsequently zero.
df.loc[
    (df['Mas Vnr Type'] == 'None') &
    (df['Mas Vnr Area'] != 0),
    'Mas Vnr Area'
] = 0

# If Garage area is 0 then the equivalent column ordinal/nominal shall also be None/0.
df['Garage Area'] = df['Garage Area'].fillna(0)
df.loc[
    (df['Garage Area'] == 0),
    ['Garage Type', 'Garage Finish', 'Garage Qual', 'Garage Cond']
] = "None"
df['Garage Finish'] = df['Garage Finish'].fillna("None")
df['Garage Qual'] = df['Garage Qual'].fillna("None")
df['Garage Cond'] = df['Garage Cond'].fillna("None")
df['Garage Yr Blt'] = df['Garage Yr Blt'].fillna(0)
df.loc[
    (df['Garage Area'] == 0),
    'Garage Cars'
] = 0

df['Lot Frontage'] = (
    df['Lot Frontage'].fillna(df.groupby('Neighborhood')['Lot Frontage'].transform('median'))
)

df['Lot Frontage'] = df['Lot Frontage'].fillna(df['Lot Frontage'].mode()[0])

- Fill NaN with None
- For Mas Vnr Type Set the equivalent column value Mas Vnr Type to Zero as well.
- For Bsmt if Total Bsmt SF is 0 then the equivalent Basement Columns shall be set to None (Ordinal) or Zero (Nominal)
- Electrical feature only has one missing value set that to mode of any.
- Garage Type replace with None moreover, The equivalent features that do not have any values shall be dropped or filled with Mode.